# 06 — LoRA fine-tuning as a controlled change

**Estimated time:** 55 minutes<br>
**Prerequisites:** 05 — Prompt baselines<br>
**Learner-produced evidence:** a reviewed training configuration and measured adapter evidence

## Learning objectives

- Explain which weights LoRA changes and which base artifacts remain fixed.
- Connect the portable chat records to the MLX-LM training configuration.
- Interpret loss and memory as training evidence, not adoption evidence.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

Full fine-tuning changes every selected model weight and is often unnecessary for a narrow local task. LoRA learns a much smaller update while keeping the base model frozen, which reduces trainable parameters and makes adapters portable. It still creates a new model behavior that needs exact lineage, validation, resource measurement, and frozen evaluation.

## Key terms in plain language

- **parameter / weight:** a learned numeric value used by the neural network to transform inputs.
- **gradient:** the direction and magnitude by which optimization proposes changing trainable parameters.
- **loss:** the numerical training objective minimized over examples; it is a proxy, not the product decision.
- **iteration:** one optimizer update; it may consume one batch or an accumulated set of micro-batches.
- **batch:** examples processed together before an optimizer update or gradient-accumulation step.
- **quantization:** representing model values with lower precision to reduce memory and sometimes compute cost.
- **LoRA rank:** the size of the low-dimensional learned update; larger is more expressive but trains more parameters.
- **adapter:** the learned LoRA configuration and weights that must be paired with the compatible base checkpoint.
- **checkpoint:** a saved training state or adapter snapshot from a particular step.
- **overfitting:** improving on training examples while generalization to independent examples stagnates or worsens.


## Mental model — how to think about this

Imagine the frozen base model as a large reference book and LoRA as a small transparent correction sheet placed over selected pages. Training writes only the correction sheet. Inference combines both. The sheet is useful only with the exact compatible book, and a neater-looking sheet (lower loss) still has to pass the same independent exam as every other change.

### Running example

During LoRA training, the base model remains fixed while small adapter matrices learn from records such as the password example. The adapter may learn the mapping more consistently, but decreasing loss only shows progress on its objective. The locked adapter still has to beat the prompt and rule baselines on independent data.

### Questions to ask before continuing

- What measured gap remains after the strongest deterministic and prompt baselines?
- Which base revision, quantization, layers, rank, and training data define this adapter?
- Do validation loss and task metrics suggest useful learning or overfitting?
- Is the added adapter operational burden justified by a frozen-evaluation improvement?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Pin the exact base revision and training configuration.** An adapter is not reproducible or safely loadable without its model, tokenizer, quantization, data, seed, and library versions.
- **Run a smoke train first.** Prove parsing, batching, checkpoint paths, validation, and artifact capture with a few iterations before committing time and battery to a full run.
- **Save adapters separately and immutably.** Use run-specific paths and never overwrite the canonical canonical adapter from an exploratory notebook cell.
- **Track training and validation evidence.** Log losses, steps, effective batch configuration, duration, memory context, and checkpoints; then use task evaluation for the actual decision.
- **Name quantized methods precisely.** In MLX-LM, training a quantized model uses its QLoRA path; do not use `QLoRA` as a generic label for every low-bit adapter workflow in other tools.

## Common mistakes and why they fail

- **Calling lower training loss a win.** It does not establish generalization, parseability, safety, or usefulness.
- **Silently changing the base model.** Adapters are coupled to architecture and revision; mismatches can fail or mislead.
- **Full-tuning by default on constrained hardware.** It adds memory, storage, and catastrophic-forgetting risk without first proving that a parameter-efficient change is insufficient.
- **Evaluating on training examples.** That measures memorization opportunity, not behavior on independent inputs.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [Apple MLX-LM fine-tuning with LoRA or QLoRA](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md)
- **Tool guidance:** [Hugging Face PEFT LoRA conceptual guide](https://huggingface.co/docs/peft/main/conceptual_guides/lora)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Inspect the change before running it

The base model revision and 4-bit weights stay fixed. LoRA learns a small
adapter over selected projections. The configuration is versioned so the
change can be reproduced and hashed with its later evaluation evidence.
MLX-LM calls LoRA training over this quantized base **QLoRA**. The YAML's
`fine_tune_type: lora` names the adapter type; the local model's 4-bit
weights determine that quantized training path.


In [ ]:
import json

import yaml

from aai_local_finetuning.settings import PROJECT_ROOT, load_settings

settings = load_settings()
config_path = PROJECT_ROOT / "configs" / "training" / "lora.yaml"
training_config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
training_config

## Read one portable training example

MLX-LM consumes the same framework-neutral messages a future trainer can
consume. Metadata is evidence and slicing context; it is not appended to
the user prompt as a shortcut to the target.


In [ ]:
first_training_record = json.loads(
    (settings.processed_dir / "train.jsonl").read_text(encoding="utf-8").splitlines()[0]
)
{
    "example_id": first_training_record["example_id"],
    "message_roles": [message["role"] for message in first_training_record["messages"]],
    "user_preview": first_training_record["messages"][1]["content"][:140],
    "assistant_target": json.loads(first_training_record["messages"][2]["content"]),
    "metadata_keys": sorted(first_training_record["metadata"]),
}

## What the conservative settings buy us

Batch size 1, gradient accumulation, eight adapted layers, a bounded
sequence length, prompt masking, and checkpointing reduce unified-memory
pressure. They are design choices for the prepared 24 GB Apple-silicon
machine, not universal optimal values. `grad_checkpoint` recomputes
intermediate activations to save memory; it is different from a saved
adapter checkpoint. With batch size 1 and four accumulation steps, the
approximate effective batch is four sequences per optimizer update.


In [ ]:
training_anatomy = {
    key: training_config.get(key)
    for key in (
        "model",
        "train",
        "fine_tune_type",
        "optimizer",
        "num_layers",
        "batch_size",
        "grad_accumulation_steps",
        "learning_rate",
        "max_seq_length",
        "mask_prompt",
        "grad_checkpoint",
        "iters",
        "steps_per_eval",
        "save_every",
        "seed",
    )
}
training_anatomy["method"] = "QLoRA (LoRA over a 4-bit base in MLX-LM)"
training_anatomy["approximate_effective_batch_sequences"] = (
    training_config["batch_size"] * training_config["grad_accumulation_steps"]
)
training_anatomy["lora_parameters"] = training_config.get("lora_parameters")
training_anatomy

## Load measured preflight evidence

Flight preparation already runs one real iteration to catch MLX compile,
data-shape, and memory surprises. This is a readiness probe, not the final
change. A missing file tells you preparation did not complete.


In [ ]:
preflight_path = PROJECT_ROOT / "artifacts" / "training" / "preflight-smoke.json"
preflight_evidence = (
    json.loads(preflight_path.read_text(encoding="utf-8"))
    if preflight_path.is_file()
    else {"status": "missing; prepare this machine online"}
)
preflight_evidence

## Optional live training cell

The default is safe for Run All. Set `RUN_TRAINING = True` to run ten
iterations into a notebook-specific adapter directory. This never
overwrites the canonical `bitext-lora-v1` change. For the full configured
run, set `TRAINING_ITERATIONS = None` only after the smoke evidence looks
healthy and you have enough time and battery. Immediately before MLX-LM
starts, the cell rechecks both the flight-preparation manifest and the
content-addressed dataset manifest. The flight manifest binds the governed
`src/` package, canonical notebook renderer/pedagogy source, interpreter,
platform, and exact installed package versions. A source, package, or split
change after preparation therefore fails closed instead of becoming an
unrecorded experiment change. The training success manifest captures that
same execution contract plus the expected base-model revision and every
required model and dataset file—not merely paths from the YAML. Training
writes into a fresh staging directory; only a zero-exit
run with valid adapter outputs and durable evidence is published, with
`training-manifest.json` acting as the success token. An exclusive
per-adapter lock serializes training and publication, so two terminals
cannot splice different generations together.


In [ ]:
from aai_local_finetuning.data import require_valid_manifest
from aai_local_finetuning.offline import verify_flight_manifest
from aai_local_finetuning.training import run_lora

RUN_TRAINING = False
TRAINING_ITERATIONS = 10
notebook_adapter = PROJECT_ROOT / "artifacts" / "notebook" / "adapters" / "bitext-smoke"
if RUN_TRAINING:
    verify_flight_manifest(settings)
    require_valid_manifest(PROJECT_ROOT / "data" / "processed" / "bitext-v1")
    training_evidence = run_lora(
        iterations=TRAINING_ITERATIONS,
        adapter_path=notebook_adapter,
        log_name="notebook-bitext-smoke",
    ).model_dump(mode="json")
else:
    training_evidence = {
        "status": "skipped",
        "how_to_run": "Set RUN_TRAINING = True",
        "preflight": preflight_evidence,
    }
training_evidence

## Exercise — interpret optimization evidence

After a run, compare training and validation losses and measured peak
memory. Success means your conclusion avoids claiming that lower loss
proves better frozen-test behavior or safer responses. One or ten
iterations prove plumbing, not a trustworthy loss trend. A formal run
should also record the selected checkpoint, adapter hash, base revision,
configuration hash, seed, and whether examples were truncated at 512 tokens.


In [ ]:
optimization_conclusion = (
    "The adapter optimization executed locally within the measured memory "
    "profile. Only the frozen structured-output evaluation can determine "
    "whether the change should be adopted."
)
assert "frozen" in optimization_conclusion.lower()
optimization_conclusion

**Hint:** loss is calculated on the training objective. Promotion asks a
broader question about generalization, schema, labels, policy, latency,
tokens, memory, and the strongest meaningful baseline.


## Checkpoint

You can now name the adapter as the change and explain why a successful
training process is necessary but insufficient evidence.

**Next:** `07_frozen_evaluation.ipynb` opens the frozen test once and
applies the already locked methods.
